# Original rover body/power R01

Prepared portable notebook, not yet executed in Colab. These are original manufacturing-development solids, not a qualified vehicle or detailed supplier electronics CAD. Supplier envelopes are kept in a separate file. The embedded reference frame is original R06 engineering CAD. No restricted motor CAD is included.

Running all cells installs CadQuery from the normal Python package registry, reconstructs the original panels and hardware, checks interference against the original frame, then produces STEP/GLB and evidence. Colab may require restarting the runtime after the dependency cell. Native CAD is millimetres; GLB is metres, Z-up.


In [ ]:
%pip install -q cadquery==2.7.0 "numpy>=1.26,<3" "matplotlib>=3.8,<4"


In [ ]:
from pathlib import Path
import base64
P=Path('/content/body_power_R01')
P.mkdir(exist_ok=True)
(P/'build_body_power.py').write_text('"""Original, parametric body integration R01. Native CAD mm, X forward/Y left/Z up.\nThis is engineered manufacturing-development geometry, NOT a qualified rover design.\nSupplier dimensional reservations are exported separately from original component CAD.\nNo supplier CAD geometry is copied. See README.md and SOURCES.json.\n"""\nfrom pathlib import Path\nimport cadquery as cq, numpy as np\nimport json, math, struct, csv, hashlib\nP=Path(__file__).resolve().parent\nREFERENCE_FRAME=P/\'reference_frame_R06.step\'\nPARTS=[]; PROXIES=[]\nMATS={\n \'aluminium\':{\'density\':2700,\'color\':[.57,.54,.46,1],\'assumption\':\'5083-like aluminium density; alloy/temper and weld procedure not qualified\'},\n \'lid\':{\'density\':2700,\'color\':[.66,.62,.53,1],\'assumption\':\'5083-like aluminium density; alloy/temper TBD\'},\n \'steel\':{\'density\':7850,\'color\':[.21,.25,.28,1],\'assumption\':\'generic carbon steel density; grade not specified\'},\n \'fastener\':{\'density\':7850,\'color\':[.66,.69,.70,1],\'assumption\':\'simplified unthreaded steel fastener, strength class TBD\'},\n \'gasket\':{\'density\':1200,\'color\':[.055,.064,.07,1],\'assumption\':\'EPDM-like nominal density, material and compression qualification pending\'},\n \'restraint\':{\'density\':2700,\'color\':[.76,.43,.10,1],\'assumption\':\'aluminium, unqualified strength\'},\n \'proxy\':{\'density\':0,\'color\':[.89,.45,.10,.45],\'assumption\':\'nonphysical dimensional envelope, not supplier CAD\'}\n}\ndef box(l,w,h,c):return cq.Workplane(\'XY\').box(l,w,h).translate(c).val()\ndef plate(l,w,t,z,cx=0,cy=0):return box(l,w,t,(cx,cy,z+t/2))\ndef cyl(r,h,c,axis=\'Z\'):\n wp=cq.Workplane({\'Z\':\'XY\',\'X\':\'YZ\',\'Y\':\'XZ\'}[axis]).circle(r).extrude(h)\n # XZ extrudes -Y by CadQuery\'s plane convention.\n if axis==\'Y\':wp=wp.translate((0,h,0))\n return wp.translate(c).val()\ndef add(n,s,mat=\'aluminium\',group=\'body\',note=\'\'):\n assert s.isValid(),n\n PARTS.append({\'name\':n,\'shape\':s,\'material\':mat,\'group\':group,\'note\':note})\n return s\ndef holes(s,pts,r,axis=\'Z\',start=-200,length=800):\n for a,b in pts:\n  c=(a,b,start) if axis==\'Z\' else (start,a,b) if axis==\'X\' else (a,start,b)\n  s=s.cut(cyl(r,length,c,axis))\n return s\ndef ring_rect(l,w,band,t,z,cx=0,cy=0):return plate(l,w,t,z,cx,cy).cut(plate(l-2*band,w-2*band,t+2,z-1,cx,cy))\ndef screw(n,x,y,z,shaft_len=15,r=3,head_r=5,head_h=4,group=\'fasteners\'):\n # Smooth shank and socket head are deliberate simplified fastener geometry, not vendor CAD.\n s=cyl(r,shaft_len,(x,y,z-shaft_len)).fuse(cyl(head_r,head_h,(x,y,z)))\n socket=cq.Workplane(\'XY\').polygon(6,head_r*.95).extrude(2).translate((x,y,z+head_h-2)).val()\n add(n,s.cut(socket),\'fastener\',group,\'Nominal M6-style envelope, no thread model or strength rating.\')\ndef hatch(n,xlo,xhi,ylo,yhi,z=337,group=\'hatches\'):\n l=xhi-xlo;w=yhi-ylo;cx=(xlo+xhi)/2;cy=(ylo+yhi)/2\n pts=[(xlo+14,ylo+14),(xhi-14,ylo+14),(xlo+14,yhi-14),(xhi-14,yhi-14),\n      (cx,ylo+14),(cx,yhi-14)]\n rim=holes(ring_rect(l,w,20,3,z,cx,cy),pts,3.3)\n gasket=holes(ring_rect(l-2,w-2,16,2,z+3,cx,cy),pts,3.3)\n lid=holes(plate(l-2,w-2,3,z+5,cx,cy),pts,3.3)\n # Two recessed finger pull slots: actual holes, not fake handles.\n for hx in [cx-l*.25,cx+l*.25]:\n  slot=cq.Workplane(\'XY\').center(hx,cy).slot2D(80,18,0).extrude(5).translate((0,0,z+4)).val();lid=lid.cut(slot)\n add(n+\'_weld_flange\',rim,\'aluminium\',group)\n add(n+\'_gasket\',gasket,\'gasket\',group,\'Concept gasket with 2 mm nominal uncompressed thickness; finger holes mean lid is not weather sealed.\')\n add(n+\'_removable_lid\',lid,\'lid\',group,\'3 mm laser-cut lid. Finger slots require sealed insert handles before weatherproofing.\')\n for i,(x,y) in enumerate(pts):screw(n+f\'_M6_{i+1}\',x,y,z+8,shaft_len=11,group=group)\n # Backing nut strips are separate with clearance bores. Inserts/threads not modeled.\n for i,(x,y) in enumerate(pts):\n  s=plate(18,18,4,z-4,x,y).cut(cyl(3.3,6,(x,y,z-5)))\n  add(n+f\'_nutplate_{i+1}\',s,\'steel\',group,\'M6 rivet-nut or weld-nut finishing operation not yet selected; clearance hole modeled.\')\n return (l,w)\ndef panel_vents(s,xcenter,zcenter,y,rows=5,cols=7,pitchx=24,pitchz=18):\n for i in range(cols):\n  for j in range(rows):\n   x=xcenter+(i-(cols-1)/2)*pitchx;z=zcenter+(j-(rows-1)/2)*pitchz\n   s=s.cut(box(16,12,6,(x,y,z)))\n return s\n\ndef build():\n # Main enclosure base contacts the existing rails and crossmembers at Z=100.\n base=plate(2450,1026,3,100,-675)\n base=base.cut(plate(650,720,6,99,-800))\n for xc in (-1350,0):\n  for sy in (-460,460):base=base.cut(box(410,144,10,(xc,sy,102)))\n # Slots accept proposed rail clamps, no holes silently added to inherited frame.\n clamp_pts=[(x,y) for x in (-1050,-550) for y in (-390,390)]\n base=holes(base,clamp_pts,5.4)\n add(\'main_base_deck_3mm\',base,\'aluminium\',\'main_body\',\'2.45 m laser-cut base; recessed battery-well opening; sits on frame Z=100.\')\n # Full side sheets, controller-bay vents and service access holes.\n for sign,label in [(1,\'left\'),(-1,\'right\')]:\n  y=sign*514.5\n  s=box(2450,3,117,(-675,y,161.5)) # Z103..220, Y513..516\n  s=panel_vents(s,-1525,182,y,rows=3)\n  for xc in (-1050,-550):s=s.cut(box(84,12,9,(xc,y,106.5)))\n  for xc in (-1350,0):\n   s=s.cut(box(410,14,32,(xc,y,116)))\n   s=s.cut(box(160,32,110,(xc,y,160)))\n  # Side access door cutout for terminal-service bay and forward tool compartment.\n  for xc,ll in [(-800,480),(390,250)]:s=s.cut(box(ll,10,66,(xc,y,184)))\n  add(\'main_\'+label+\'_side_3mm\',s,\'aluminium\',\'main_body\')\n  # Removable side service covers with 2 mm gaps, actual fastener holes along normal Y.\n  for j,(xc,ll) in enumerate([(-800,480),(390,250)]):\n   cover=box(ll-4,3,62,(xc,sign*514.5,184))\n   pts=[(xc+dx,184+dz) for dx in [-(ll/2-18),(ll/2-18)] for dz in [-20,20]]\n   cover=holes(cover,pts,3.3,axis=\'Y\',start=-600,length=1200)\n   # A horizontal finger recess through each plate (must seal for environmental qualification).\n   cover=cover.cut(box(76,10,12,(xc,sign*514.5,190)))\n   add(f\'{label}_side_service_cover_{j+1}\',cover,\'lid\',\'side_service\',\'62 mm-high removable access panel. Requires sealed handle inserts.\')\n   # Separate 20x20 backing clips behind access opening, not fake painted bolts.\n   for k,(x,z) in enumerate(pts):\n    clip=box(40,3,40,(x,sign*511.5,z))\n    clip=holes(clip,[(x,z)],3.3,axis=\'Y\',start=-600,length=1200)\n    add(f\'{label}_service_nutclip_{j+1}_{k+1}\',clip,\'steel\',\'side_service\',\'M6 nut/insert not modeled; clip requires weld detail.\')\n # True 3 mm flared lower and tapered upper shoulders, supported by shaped bulkheads.\n # At wheel stations underside is above Z218 mm, nominally >23 mm over provisional shock envelope.\n for sign,label in [(1,\'left\'),(-1,\'right\')]:\n  for tag,y1,z1,y2,z2 in [(\'lower\',516,220,650,260),(\'upper\',650,260,450,337)]:\n   # Plane across Y/Z; define orientation from ordered endpoints to preserve normal thickness.\n   dy=y2-y1;dz=z2-z1;ang=math.degrees(math.atan2(dz,sign*dy))\n   sh=box(2450,math.hypot(dy,dz),3,(0,0,0)).rotate((0,0,0),(1,0,0),ang).translate((-675,sign*(y1+y2)/2,(z1+z2)/2))\n   add(\'main_\'+label+\'_\'+tag+\'_sloping_shoulder\',sh,\'lid\',\'main_body\',\'Original 3mm planar welded flared shoulder; 1.3m max hull span is a design choice, not a recovered reference dimension.\')\n # End walls, with connector openings at the arm-facing forward face.\n for x,label in [(-1898.5,\'rear\'),(548.5,\'arm_facing\')]:\n  s=cq.Workplane(\'YZ\').polyline([(-513,103),(513,103),(513,220),(650,260),(450,337),(-450,337),(-650,260),(-513,220)]).close().extrude(3).translate((x-1.5,0,0)).val()\n  if x>0:s=holes(s,[(-300,235),(300,235)],25,axis=\'X\',start=540,length=20)\n  else:s=holes(s,[(0,200)],16,axis=\'X\',start=-1910,length=30)\n  add(label+\'_bulkhead_3mm\',s,\'aluminium\',\'main_body\',\'Connector apertures only; glands, EMC seals, bonding and connectors not selected.\')\n # Internal segregating bulkheads. Cable routing openings have true B-rep material removal.\n for x,label in [(-1141.5,\'controller_battery\'),(-458.5,\'battery_tool\')]:\n  s=cq.Workplane(\'YZ\').polyline([(-513,103),(513,103),(513,220),(650,260),(450,337),(-450,337),(-650,260),(-513,220)]).close().extrude(3).translate((x-1.5,0,0)).val()\n  s=holes(s,[(300,268)],22,axis=\'X\',start=x-5,length=10)\n  s=s.cut(box(10,75,55,(x,440,260)))\n  add(label+\'_partition\',s,\'aluminium\',\'main_body\',\'Battery/cable segregation with service openings; fire barrier not qualified.\')\n # Recessed welded battery well through the base deck. Corners meet at seams, no overlaps.\n well_floor=plate(650,720,3,-80,-800)\n well_floor=holes(well_floor,[(x,y) for x in (-950,-650) for y in (-235,235)],4.4)\n add(\'battery_well_floor_3mm\',well_floor,\'aluminium\',\'battery_well\')\n for y,label in [(358.5,\'left\'),(-358.5,\'right\')]:\n  s=box(650,3,414,(-800,y,130)) # -77..337\n  if y>0:s=holes(s,[(-900,268)],22,axis=\'Y\',start=350,length=20)\n  add(\'battery_well_\'+label+\'_wall\',s,\'aluminium\',\'battery_well\')\n for x,label in [(-1123.5,\'rear\'),(-476.5,\'front\')]:\n  s=box(3,714,414,(x,0,130))\n  add(\'battery_well_\'+label+\'_wall\',s,\'aluminium\',\'battery_well\')\n # Exact 4-pack support pads and machined/laser-cut restraints. No battery detail invented.\n for i,(x,y) in enumerate([(x,y) for x in (-950,-650) for y in (-110,110)]):\n  add(f\'battery_{i+1}_isolator_pad\',plate(260,180,4,-77,x,y),\'gasket\',\'battery_support\',\'4 mm pad. Battery mass/thermal/chemical compatibility requires supplier validation.\')\n  # Low lateral stop angles, 10 mm clearance from the dimensional battery envelope.\n  for sign in (-1,1):\n   yy=y+sign*104\n   add(f\'battery_{i+1}_stop_{sign}\',box(260,3,28,(x,yy,-59)),\'restraint\',\'battery_support\',\'Welded low stop; supplier hold-down interface unknown.\')\n  PROXIES.append({\'name\':f\'DIMENSION_ONLY_RELiON_InSight_48V_battery_{i+1}\',\'shape\':box(260,180,276,(x,y,65)),\'material\':\'proxy\',\'mass_kg\':15.6,\'group\':\'battery_envelopes\',\'note\':\'Supplier overall dimensions only; terminal and handle details NOT modeled.\'})\n # Top-restraint crossbars intentionally clear nominal envelope by 12 mm. Straps are adjustable concept.\n for i,x in enumerate((-950,-650)):\n  s=plate(30,500,4,215,x,0)\n  s=holes(s,[(x,-235),(x,235)],4.4)\n  add(f\'battery_retention_crossbar_{i+1}\',s,\'restraint\',\'battery_support\',\'12 mm top gap reserved for adjustable insulating pads; clamps NOT contact-qualified.\')\n  for sy in (-235,235):\n   add(f\'battery_retention_stud_{i+1}_{sy}\',cyl(4,313,(x,sy,-87)),\'fastener\',\'battery_support\',\'Smooth M8 tie-rod approximation through real floor/crossbar clearance bores; terminal clearance and preload unresolved.\')\n   for zz,tag in [(-87,\'bottom\'),(219,\'top\')]:\n    nut=cq.Workplane(\'XY\').polygon(6,15).circle(4.25).extrude(7).translate((x,sy,zz)).val()\n    add(f\'battery_retention_nut_{i+1}_{sy}_{tag}\',nut,\'fastener\',\'battery_support\',\'M8 hex-nut approximation; smooth clearance bore represents omitted threads.\')\n # Top deck support ledges, three independent service lids.\n # Rear controller hatch; four controllers/electronics areas do not fill the payload bay.\n hatch(\'rear_controller_hatch\',-1900,-1144,-450,450)\n # Annular shoulder covers the difference between battery well and external body width.\n batt_shoulder=plate(680,900,3,337,-800).cut(plate(650,720,5,336,-800))\n add(\'battery_deck_shoulder\',batt_shoulder,\'lid\',\'battery_hatch\')\n hatch(\'battery_hatch\',-1125,-475,-360,360,337,\'battery_hatch\')\n hatch(\'payload_tool_hatch\',-456,550,-450,450,337,\'tool_hatch\')\n # Two OEM DC controller mounting trays with slotted rails; dimensions-only electronics separate.\n for i,x in enumerate((-1710,-1370)):\n  tr=plate(270,550,3,140,x)\n  # universal 8x24 slots, deliberately not asserted as vendor bolt pattern\n  for xx in (x-105,x+105):\n   for yy in (-235,235):tr=tr.cut(cq.Workplane(\'XY\').center(xx,yy).slot2D(24,8,0).extrude(5).translate((0,0,139)).val())\n  add(f\'controller_mount_tray_{i+1}\',tr,\'aluminium\',\'controller_trays\',\'Adapter slot locations are our design, not verified against UR OEM hole drawing.\')\n  for sy in (-250,250):\n   # Fold-free built-up L stand-offs, separately modelled weldment plates.\n   add(f\'controller_tray_leg_{i+1}_{sy}\',box(250,3,37,(x,sy,121.5)),\'aluminium\',\'controller_trays\')\n  PROXIES.append({\'name\':f\'DIMENSION_ONLY_UR_OEM_DC_controller_{i+1}\',\'shape\':box(168,451,150,(x,0,218)),\'material\':\'proxy\',\'mass_kg\':4.3,\'group\':\'controller_envelopes\',\'note\':\'451×150×168 mm dimensions; source axes rotated so width is Y. No supplier mounting holes or connector detail.\'})\n # Segregated cable trough on left (positive Y) side aligns with actual partition openings. 60mm clear width.\n # Tray stays above the rail; interruptions at compartments preserve removable service.\n for i,(xc,ll) in enumerate([(-1518,690),(-800,650),(40,960)]):\n  add(f\'cable_duct_floor_{i+1}\',plate(ll,70,2,230,xc,435),\'aluminium\',\'cable_route\')\n  for yy in (400,470):\n   ds=box(ll,2,40,(xc,yy,252))\n   if i==1 and yy==400:ds=ds.cut(box(64,8,30,(-900,yy,259)))\n   add(f\'cable_duct_side_{i+1}_{yy}\',ds,\'aluminium\',\'cable_route\',\'Open-top duct, branch notch aligned to battery cable aperture; cable/strain-relief selection pending.\')\n # Strap clamps pass outside rail Y410..510; no holes in inherited rails.\n for i,(x,sign) in enumerate([(x,s) for x in (-1050,-550) for s in (-1,1)]):\n  y=sign*460;pts=[(x,sign*390),(x,sign*530)]\n  top=holes(plate(80,160,6,103,x,y),pts,5.4)\n  # At inner bolt the base deck already includes a clearance bore.\n  add(f\'rail_saddle_{i+1}_top\',top,\'steel\',\'rail_clamps\',\'80×160×6 strap, clamp preload and frame wall crush checks pending.\')\n  bot=holes(plate(80,160,6,-106,x,y),pts,5.4)\n  add(f\'rail_saddle_{i+1}_bottom\',bot,\'steel\',\'rail_clamps\')\n  for j,(xx,yy) in enumerate(pts):\n   screw(f\'rail_saddle_{i+1}_M10_{j+1}\',xx,yy,109,shaft_len=226,r=5,head_r=8.5,head_h=7,group=\'rail_clamps\')\n   nut=cq.Workplane(\'XY\').polygon(6,19.63).circle(5.25).extrude(10).translate((xx,yy,-116)).val()\n   add(f\'rail_saddle_{i+1}_M10_nut_{j+1}\',nut,\'fastener\',\'rail_clamps\',\'M10 hex-nut approximation, smooth clearance bore; strength class/preload not selected.\')\n # Front tapered tool/nose enclosure, kept clear of both mounting plates.\n # This is a 3mm welded sheet module with sloped crash-deflecting hood, not an impact-rated bumper.\n fb=plate(750,1026,3,100,1575)\n for sy in (-460,460):fb=fb.cut(box(410,144,10,(1350,sy,102)))\n add(\'front_tool_base\',fb,\'aluminium\',\'front_tool\')\n rw=box(3,1026,194,(1201.5,0,200))\n for sy in (-460,460):rw=rw.cut(box(10,144,32,(1201.5,sy,116)))\n add(\'front_tool_rear_wall\',rw,\'aluminium\',\'front_tool\')\n # Side plates: XZ plane extrudes -Y. Flat 3mm blanks with actual trapezoid profile.\n poly=[(1203,103),(1947,103),(1947,187),(1700,297),(1203,297)]\n for sign,label in [(1,\'left\'),(-1,\'right\')]:\n  s=cq.Workplane(\'XZ\').polyline(poly).close().extrude(3).val()\n  s=s.translate((0,516 if sign==1 else -513,0))\n  s=s.cut(box(410,14,32,(1350,sign*514.5,116)))\n  s=s.cut(box(160,32,110,(1350,sign*514.5,160)))\n  add(\'front_tool_\'+label+\'_side\',s,\'aluminium\',\'front_tool\')\n add(\'front_tool_nose_wall\',box(3,1026,84,(1948.5,0,145)),\'aluminium\',\'front_tool\')\n # Solid normal-thickness sloped panel between X1700/Z297 and X1947/Z187.\n dx=247;dz=-110;length=math.hypot(dx,dz);ang=math.degrees(math.atan2(-dz,dx))\n slope=box(length,1026,3,(0,0,0)).rotate((0,0,0),(0,1,0),ang).translate(((1700+1947)/2,0,(297+187)/2+1.5))\n add(\'front_sloped_service_nose\',slope,\'lid\',\'front_tool\',\'3mm planar sloped sheet; final seam trim and weld detail still required.\')\n hatch(\'front_tool_hatch\',1203,1697,-513,513,297,\'front_tool\')\n # Low removable arm station infill stays below robot mounting contact plane.\n infill=plate(646,200,3,100,875,0)\n add(\'arm_center_cable_guard\',infill,\'aluminium\',\'arm_service\',\'Between 400mm arm mounting plates, does not cover UR bolt patterns.\')\n # Notch edge guards end before arm pads; nothing is raised above Z130 in the arm station.\n # Chassis-end bumper tubes with real hollow section and cap sheets.\n for x,label in [(-2025,\'rear\'),(2025,\'front\')]:\n  tube=box(50,1090,70,(x,0,-10)).cut(box(44,1092,64,(x,0,-10)))\n  add(label+\'_bumper_RHS_70x50x3\',tube,\'steel\',\'bumpers\',\'Development guard, not crash/tow rated. Attachment brackets must be load-qualified.\')\n  for sy in (-1,1):\n   add(label+f\'_bumper_cap_{sy}\',box(50,3,70,(x,sy*546.5,-10)),\'steel\',\'bumpers\')\n  for sy in (-460,460):\n   # brackets meet frame end plane X±2000 and bumper inner wall X±2000\n   bracket=box(30,70,6,(x+(-1 if x>0 else 1)*10,sy,26))\n   # Tube inner face meets existing rail end directly. Brackets omitted to avoid unqualified material overlap.\n\n\ndef finish_seams():\n # C02 conservative corner clamp/clevis cutouts. Original panels only, never alter reference frame.\n for p in PARTS:\n  for xx in (-1350,0,1350):\n   for sy in (-1,1):\n    cut=box(480,150,33,(xx,sy*460,115.5)) # Xstation+/-240, |Y|385..535, Z99..132\n    pb=p[\'shape\'].BoundingBox();cb=cut.BoundingBox()\n    if not (pb.xmax<cb.xmin or pb.xmin>cb.xmax or pb.ymax<cb.ymin or pb.ymin>cb.ymax or pb.zmax<cb.zmin or pb.zmin>cb.zmax):p[\'shape\']=p[\'shape\'].cut(cut)\n # Explicit butt-joint trims at shoulder/side/bulkhead intersections, no hidden material overlaps.\n for p in PARTS:\n  if p[\'name\'].endswith(\'sloping_shoulder\'):\n   for other in PARTS:\n    if (p[\'name\'].endswith(\'upper_sloping_shoulder\') and other[\'name\']==p[\'name\'].replace(\'upper_sloping\',\'lower_sloping\')) or other[\'name\'] in [\'main_left_side_3mm\',\'main_right_side_3mm\',\'rear_bulkhead_3mm\',\'arm_facing_bulkhead_3mm\',\'controller_battery_partition\',\'battery_tool_partition\']:\n     p[\'shape\']=p[\'shape\'].cut(other[\'shape\'])\n   p[\'note\']+=\' Edge mitres and bulkhead notches explicitly trimmed in B-rep.\'\n\ndef bounds(s):\n from OCP.Bnd import Bnd_Box\n from OCP.BRepBndLib import BRepBndLib\n b=Bnd_Box();BRepBndLib.AddOptimal_s(s.wrapped,b,False,False);return list(b.Get())\ndef export_glb(parts,path):\n data=bytearray();views=[];acc=[];meshes=[];nodes=[];mat_names=list(MATS)\n def buf(a,typ,comp):\n  a=np.asarray(a,dtype=np.float32 if comp==5126 else np.uint32)\n  data.extend(b\'\\0\'*((-len(data))%4));off=len(data);data.extend(a.tobytes());vi=len(views);views.append({\'buffer\':0,\'byteOffset\':off,\'byteLength\':a.nbytes});idx=len(acc)\n  ac={\'bufferView\':vi,\'componentType\':comp,\'count\':len(a),\'type\':typ}\n  if typ==\'VEC3\':ac.update(min=a.min(axis=0).tolist(),max=a.max(axis=0).tolist())\n  acc.append(ac);return idx\n for p in parts:\n  s=p[\'shape\'];verts,tris=s.tessellate(.35,.12);v=np.array([[q.x,q.y,q.z] for q in verts])/1000;f=np.asarray(tris,np.uint32);v=v[f].reshape(-1,3)\n  no=np.cross(v[1::3]-v[::3],v[2::3]-v[::3]);no/=np.maximum(np.linalg.norm(no,axis=1)[:,None],1e-30);no=np.repeat(no,3,axis=0)\n  pi=buf(v,\'VEC3\',5126);ni=buf(no,\'VEC3\',5126);ii=buf(np.arange(len(v),dtype=np.uint32),\'SCALAR\',5125)\n  meshes.append({\'name\':p[\'name\'],\'primitives\':[{\'attributes\':{\'POSITION\':pi,\'NORMAL\':ni},\'indices\':ii,\'material\':mat_names.index(p[\'material\'])}]})\n  nodes.append({\'name\':p[\'name\'],\'mesh\':len(meshes)-1,\'extras\':{\'classification\':\'dimensional_reservation\' if p[\'material\']==\'proxy\' else \'original_engineering_CAD\',\'group\':p[\'group\'],\'note\':p[\'note\']}})\n mats=[{\'name\':k,\'pbrMetallicRoughness\':{\'baseColorFactor\':v[\'color\'],\'metallicFactor\':0 if k in [\'gasket\',\'proxy\'] else .45,\'roughnessFactor\':.6},**({\'alphaMode\':\'BLEND\',\'doubleSided\':True} if k==\'proxy\' else {})} for k,v in MATS.items()]\n doc={\'asset\':{\'version\':\'2.0\',\'generator\':\'Original body-power R01, CadQuery B-rep tessellation; metres Z-up\'},\'scene\':0,\'scenes\':[{\'nodes\':list(range(len(nodes)))}],\'nodes\':nodes,\'meshes\':meshes,\'materials\':mats,\'buffers\':[{\'byteLength\':len(data)}],\'bufferViews\':views,\'accessors\':acc,\'extras\':{\'units\':\'metres\',\'upAxis\':\'Z\',\'qualified\':False,\'classification\':\'dimensional_reservations_only\' if parts is PROXIES else \'original_manufacturing_development_CAD\'}}\n js=json.dumps(doc,separators=(\',\',\':\')).encode();js+=b\' \'*((-len(js))%4);data+=b\'\\0\'*((-len(data))%4)\n path.write_bytes(struct.pack(\'<III\',0x46546C67,2,28+len(js)+len(data))+struct.pack(\'<II\',len(js),0x4E4F534A)+js+struct.pack(\'<II\',len(data),0x004E4942)+data)\n\ndef verify():\n bs=[bounds(p[\'shape\']) for p in PARTS];inter=[];contacts=[]\n for i,a in enumerate(PARTS):\n  for j in range(i+1,len(PARTS)):\n   b=PARTS[j]\n   if any(bs[i][k+3]<bs[j][k]-1e-5 or bs[j][k+3]<bs[i][k]-1e-5 for k in range(3)):continue\n   vol=a[\'shape\'].intersect(b[\'shape\']).Volume()\n   if vol>1e-3:inter.append({\'a\':a[\'name\'],\'b\':b[\'name\'],\'volume_mm3\':vol})\n   elif a[\'shape\'].distance(b[\'shape\'])<1e-5:contacts.append([a[\'name\'],b[\'name\']])\n frame=cq.importers.importStep(str(REFERENCE_FRAME)).val()\n frame_inter=[]\n for p in PARTS:\n  b=bounds(p[\'shape\'])\n  if b[2]>.131e3 or b[5]<-100.001:continue\n  vol=p[\'shape\'].intersect(frame).Volume()\n  if vol>1e-3:frame_inter.append({\'part\':p[\'name\'],\'volume_mm3\':vol})\n props=[]\n for p in PARTS:\n  s=p[\'shape\'];c=s.Center();mass=s.Volume()*1e-9*MATS[p[\'material\']][\'density\']\n  props.append({\'name\':p[\'name\'],\'group\':p[\'group\'],\'material\':p[\'material\'],\'valid\':s.isValid(),\'solid_count\':len(s.Solids()),\'mass_kg\':mass,\'volume_m3\':s.Volume()*1e-9,\'com_m\':[c.x/1000,c.y/1000,c.z/1000],\'bounds_m\':(np.asarray(bounds(s))/1000).tolist(),\'note\':p[\'note\']})\n mass=sum(p[\'mass_kg\'] for p in props);cg=(sum((np.asarray(p[\'com_m\'])*p[\'mass_kg\'] for p in props),np.zeros(3))/mass).tolist()\n overall=np.vstack(bs);bnd=[overall[:,:3].min(axis=0).tolist(),overall[:,3:].max(axis=0).tolist()]\n proxyprops=[];proxycoll=[]\n for q in PROXIES:\n  c=q[\'shape\'].Center();proxyprops.append({\'name\':q[\'name\'],\'mass_kg\':q[\'mass_kg\'],\'com_m\':[c.x/1000,c.y/1000,c.z/1000],\'bounds_m\':(np.asarray(bounds(q[\'shape\']))/1000).tolist(),\'note\':q[\'note\']})\n  for p in PARTS:\n   vol=q[\'shape\'].intersect(p[\'shape\']).Volume() if all(bounds(q[\'shape\'])[k]<bounds(p[\'shape\'])[k+3] and bounds(p[\'shape\'])[k]<bounds(q[\'shape\'])[k+3] for k in range(3)) else 0\n   if vol>1e-3:proxycoll.append({\'proxy\':q[\'name\'],\'part\':p[\'name\'],\'volume_mm3\':vol})\n loaded_mass=mass+sum(q[\'mass_kg\'] for q in proxyprops);loaded_cg=(np.asarray(cg)*mass+sum((np.asarray(q[\'com_m\'])*q[\'mass_kg\'] for q in proxyprops),np.zeros(3)))/loaded_mass\n report={\'revision\':\'R01\',\'release_status\':\'manufacturing-development, not fabrication/operational release\',\'units\':{\'CAD\':\'mm\',\'GLB\':\'m, Z-up\'},\'original_part_count\':len(PARTS),\'individual_valid\':all(p[\'valid\'] for p in props),\'bounds_m\':(np.asarray(bnd)/1000).tolist(),\'material_assumptions\':MATS,\'body_hardware_mass_kg\':mass,\'body_hardware_com_m\':cg,\'body_plus_reserved_component_mass_kg\':loaded_mass,\'body_plus_reserved_component_com_m\':loaded_cg.tolist(),\'positive_volume_body_intersections\':inter,\'frame_intersections\':frame_inter,\'dimensional_reservation_intersections\':proxycoll,\'zero_gap_contacts\':contacts,\'parts\':props,\'dimensional_reservations\':proxyprops,\'exclusions\':[\'frame mass, arms, drive train, wheels, tools, cabling, cooling fans, glands and fastener threads excluded from this package mass\',\'OEM controller mass allocated uniformly at dimensional-envelope centroid, not measured supplier CG\',\'battery mass allocated uniformly at supplier-envelope centroid, not measured supplier CG\',\'mounting/welding/contact validation is geometrical only; no strength, fatigue, stability or IP qualification\']}\n (P/\'verification.json\').write_text(json.dumps(report,indent=2));print(json.dumps({k:v for k,v in report.items() if k not in [\'parts\',\'zero_gap_contacts\',\'material_assumptions\',\'dimensional_reservations\']},indent=2),flush=True)\n return report\n\ndef main():\n build();finish_seams();print(\'built\',len(PARTS),\'parts\',flush=True)\n r=verify()\n for parts,stem in [(PARTS,\'body_power_R01\'),(PROXIES,\'cots_reservations_R01\')]:\n  ass=cq.Assembly(name=stem)\n  for p in parts:ass.add(p[\'shape\'],name=p[\'name\'],color=cq.Color(*MATS[p[\'material\']][\'color\']))\n  ass.export(str(P/(stem+\'.step\')));export_glb(parts,P/(stem+\'.glb\'));print(\'exported\',stem,flush=True)\n with (P/\'original_cutlist.csv\').open(\'w\') as f:\n  w=csv.writer(f);w.writerow([\'part\',\'group\',\'material_assumption\',\'mass_kg\',\'bounds_mm\',\'note\'])\n  for p in r[\'parts\']:w.writerow([p[\'name\'],p[\'group\'],MATS[p[\'material\']][\'assumption\'],p[\'mass_kg\'],json.dumps((np.asarray(p[\'bounds_m\'])*1000).tolist()),p[\'note\']])\n if r[\'positive_volume_body_intersections\'] or r[\'frame_intersections\'] or r[\'dimensional_reservation_intersections\']:print(\'FIX INTERSECTIONS BEFORE RELEASE\',flush=True)\nif __name__==\'__main__\':main()\n')
(P/'reference_frame_R06.step').write_bytes(base64.b64decode('...'))
(P/'README.md').write_text("# Body/power R01: original engineering CAD\n\nThis package replaces an exposed ladder-only appearance with functional compartment geometry. It is a manufacturing-development assembly, not fabrication-released, electrically qualified, weatherproof, crash-rated or ready to operate.\n\n## Coordinate and source contract\n\nAll original construction is parametric CadQuery B-rep in millimetres, frame C: X forward, Y left, Z up. GLB vertex positions are metres and Z-up; no root correction rotation is required. Mount origins remain the R06 frame origins. Geometry is not stretched to match photographs.\n\n- `body_power_R01.step` / `.glb`: original sheet parts, rail straps, stiffeners, hatches, gaskets, hardware approximations and cable troughs. No purchased electronics are disguised as detailed CAD.\n- `cots_reservations_R01.step` / `.glb`: six explicitly named DIMENSION_ONLY cuboids: four RELiON batteries and two UR OEM DC controllers. Load only as a separate optional translucent packaging layer.\n- `build_body_power.py`: reproducible source. `python build_body_power.py` rebuilds and verifies. CadQuery 2.7 and NumPy are required.\n- `verification.json`: current solid validity, actual bounds, geometric mass/CG assumptions and intersections. Frame, arms, wheels and drive masses are excluded.\n- `original_cutlist.csv`: one row per original part. This is a finished-geometry list, not a released flat-pattern/nesting set.\n- `sourced_bom.csv`, `SOURCES.json`: primary-source identity, dimensions, mass and limitations. No purchase or vendor approval is claimed.\n\n## Mechanically useful features\n\nThe recessed battery well sits between the existing X −1200 and −400 crossmembers, inside the rails. Four upright 260×180×276 mm reservations rest on 4 mm isolator pads; the reserved bank weighs 62.4 kg. Its original aluminium floor/walls, low stops and top crossbar provisions are explicit solids. Terminal positions and a validated hold-down contact interface remain unknown; the bars have a 12 mm nominal clearance and must not be represented as a completed restraint design.\n\nThe rear bay has two distinct elevated controller mounting trays with actual adapter slots, free connector/air volume and removable top access. The middle bay is retained as tool/payload volume. The forward compartment uses a real planar sloping nose and removable hatch. No weapon, targeting system or weapon mounting interface is included.\n\nThe outer body has separate 3 mm panels, shaped transverse bulkheads, sloping shoulder plates, four side inspection panels, top hatch joints with gasket solids, M6 clearance holes/nutplate provisions, simplified socket-head screws, cable troughs and actual connector/vent apertures. The top remains segmented for access. Rails are attached using wrap-around strap-clamp provisions, with no undisclosed new holes in the inherited frame. Fastener threads, bolt preload, rivet-nut choice, weld details, bend allowance, insulation and bonding still need engineering release.\n\nThe broad upper-hull span is a design choice suggested by non-dimensional reference-image cues. Its underside respects the current provisional suspension reservation; that does not establish swept-envelope clearance. The lower body has explicit cutouts for the current station-centred wishbone clamp reservation, at X −1350, 0 and +1350 mm. Do not use the superseded trailing-arm relief positions.\n\n## Service and access\n\nThe battery hatch removes vertically; all four battery envelopes can then lift vertically with restraints removed. The controller bay is top-accessible; full connector extraction and fan service remain to be validated against acquired supplier CAD. Side inspection panels are supplemental, not battery-extraction doors. Front and middle tool hatches are separate from arm mounting plates. The arm bases at X900, Y±300, Z130 mm retain their existing bolt patterns; arm-link clearances must be checked in the actual default and motion poses.\n\nFinger slots and ventilation openings are intentionally open geometry. The concept gaskets do not confer IP protection: sealed handles, fans/filters, drains, glands, cable penetrations and full thermal/environmental qualification are outstanding. The design must not be sold or described as a sealed battery enclosure or a certified safety system.\n\n## Power architecture limits\n\nThe four proposed batteries provide 6.144 kWh nominal arithmetic energy, not usable energy or range. Their individual ratings do not imply system current capability. The UR source lists OEM DC input 19–72 VDC, up to 400 A inrush and 350 W control-unit heat per UR20. Two controllers require verified precharge/soft-start, fuse coordination and thermal design; nominal voltage overlap is insufficient. Standard UR control boxes require AC and are not equivalent to these provisional OEM DC reservations.\n\nAt a hypothetical 15 K air rise, removing the 700 W combined controller heat bound would require roughly 0.039 m³/s ideal airflow using assumed air density 1.2 kg/m³ and heat capacity 1005 J/(kg K). This calculation excludes pressure losses, ambient extremes, drive losses, battery heat and fan derating. It is a sizing warning, not fan selection or a cooling-performance claim. Current small vent slots alone are not qualified to deliver this flow.\n\nTraction motors/controllers, regeneration, contactors, fuse ratings, BMS coordination, charger, emergency stop architecture, functional safety and grounding are all unresolved. No WD220 proprietary/restricted CAD is part of this package. No electrical connections are portrayed as a qualified system.\n\n## Verification meaning\n\nMass is computed from CAD volume using stated nominal densities. Battery/controller masses come from manufacturer data; their CGs are placed at envelope centres as assumptions. This does not validate full-vehicle CG, tip stability, axle loads, braking or manipulator reaction loads. Frame mass and structure remain a separate package and may be redesigned. Pairwise positive-volume checks are CAD interference checks, not strength, safety or manufacturability certification.\n\nThe original bumper tubes are open-ended hollow-section geometry with separate caps. They are unqualified guards, not towing/recovery points or impact-rated bumpers. No recovery rating is invented.\n\n## Traction candidate and optional second bay\n\nThe WD220 drawing lists47A within the motor block, without explicitly defining DC-bus input current. Six such ratings sum282A arithmetically; that is not verified battery demand. Inverter current/efficiency data are needed. A separate power check is possible: if six drives simultaneously deliver2.2kW mechanical output each,13.2kW/51.2V requires257.8A even at ideal100% efficiency, before arms/auxiliaries. Actual input would be higher. This exceeds the four batteries'180A summed individual recommended discharge, but does not by itself exceed the400A sum of individual maxima; neither sum is a qualified pack rating. The motor duty is S2 for60minutes, not continuous. Drive price and exact mass are unquoted/unknown. No motor/controller/battery compatibility is claimed. Controller heat is used for cooling sizing only and is not added again as independent electrical input power.\n\nA separately recessed optional four-unit bay is geometrically possible within the R06 opening between crossmembers X−400 and+750mm, provisionally at X[−300,+350],Y±360,Z[−80,+337]mm. It is not modeled or included in mass/capacity. The present tool-bay interior is234mm high and cannot hold276mm batteries above its unmodified floor. This variant requires a new lowered well, changed floor, qualified retention and loss of that tool/payload volume; electrical limits do not disappear by adding units. See reservation_checks.json.\n\n## Portable / Colab execution\n\nThe package includes the original reference_frame_R06.step solely for body/frame interference checks. Run the included requirements installation in an authorized Python/Colab environment, then build_body_power.py. Main STEP/GLB generation does not depend on private assets or network downloads. The optional arm-screen scripts require the separately licensed original robot asset folder; set its path in check_arm_clearance.py before use. This package was executed in the current engineering environment; it does not claim execution in Colab.\n\n## C02 suspension packaging update\n\nCurrent provisional wheel centres are Y±0.94m and Z−0.25m, superseding Y±0.86m. The lower corner relief is480mm wide belowZ132mm. Local160mm-wide coilover service notches extend toZ215mm. C02 shock envelopes are Xstation±60mm, |Y|520–800mm and Z−430..200mm; no actual spring has been selected. The source-dimension tyre concept is876.3mm OD and317.5mm provisional section. The analytic parallel-link, locked-steering ±12° check gives about94mm minimum lateral body-to-tyre envelope separation; this is not actual tyre/rim fit, loaded clearance or certified travel. See tire_envelope_screen_C02.json and reservation_checks.json.\n")
(P/'SOURCES.json').write_text('[\n  {\n    "id": "RELION-48V030-GC2",\n    "manufacturer": "RELiON / Navico",\n    "component": "InSight 48V 30Ah 48V030-GC2",\n    "source_url": "https://www.relionbattery.com/products/lithium/insight-gc2-48v",\n    "drawing_url": "https://relion-battery.s3.us-east-1.amazonaws.com/docs/product/InSightDataSheet_48V_240425_110052.pdf",\n    "accessed_utc": "2026-10-09",\n    "primary_source": true,\n    "dimensions_mm": [\n      260,\n      180,\n      276\n    ],\n    "mass_kg": 15.6,\n    "source_CAD_acquired": false,\n    "model_class": "original dimensional envelope only",\n    "redistribution": "No supplier CAD or artwork included. Manufacturer datasheet linked, not copied into deliverable. Original reservation cuboid encodes factual overall dimensions only.",\n    "facts": {\n      "nominal_voltage_V": 51.2,\n      "capacity_Ah": 30,\n      "energy_Wh": 1536,\n      "parallel_units_max": 10,\n      "series_allowed": false,\n      "recommended_continuous_discharge_A_each": 45,\n      "maximum_continuous_discharge_A_each": 100,\n      "peak_charge_V": 57,\n      "continuous_charge_A_each_max": 30\n    },\n    "limits": [\n      "Individual published ratings do not establish rover system limits.",\n      "Four proposed parallel units reserve 62.4kg and 6144Wh nominal. This is not a runtime claim.",\n      "Do not use golf-cart mileage claims for this rover.",\n      "Inrush, regen acceptance, BMS sharing, fusing, contactors, charging and thermal design remain unqualified.",\n      "Battery terminal/handle positions and supplier hold-down details are not modeled."\n    ]\n  },\n  {\n    "id": "UR-OEM-DC",\n    "manufacturer": "Universal Robots",\n    "component": "OEM DC control box dimensional reservation",\n    "source_url": "https://www.universal-robots.com/manuals/EN/HTML/SW5_25/Content/prod_oem_cb_all/oem_cb_all_specs_root.htm",\n    "product_url": "https://www.universal-robots.com/products/cb5-oem/",\n    "supplier_CAD_page": "https://www.universal-robots.com/download/mechanical-e-series/control-box-oem/control-box-3d-step-file-cb55cb56-oem-ac-dc-ur-series-e-series/",\n    "accessed_utc": "2026-10-09",\n    "primary_source": true,\n    "dimensions_mm_WHD": [\n      451,\n      150,\n      168\n    ],\n    "mass_kg": 4.3,\n    "source_CAD_acquired": false,\n    "model_class": "original dimensional envelope only",\n    "redistribution": "Supplier STEP available behind account gate under Graphical Documentation terms. No file downloaded, copied or republished. Reservation is our dimensional cuboid.",\n    "facts": {\n      "input_min_VDC": 19,\n      "input_typical_VDC": [\n        24,\n        48\n      ],\n      "input_max_VDC": 72,\n      "inrush_A_max": 400,\n      "external_fuse_A_max": 100,\n      "UR20_control_unit_heat_W_max": 350\n    },\n    "limits": [\n      "OEM DC controller choice is provisional; exact controller revision and robot compatibility need supplier confirmation.",\n      "A 51.2V nominal battery being inside the published input range does not qualify the power architecture.",\n      "400A inrush requires verified start-up/soft-start design; do not assume battery bank supplies both controllers at once.",\n      "350W maximum control-unit heat per UR20 means 700W for two as a thermal sizing bound, not measured operating dissipation.",\n      "Standard enclosed control boxes use AC; no claim the battery directly powers them.",\n      "No vendor mounting hole positions, connector geometry, cooling clearances or orientations inferred."\n    ]\n  },\n  {\n    "id": "VIC-ORION-CANDIDATE",\n    "manufacturer": "Victron Energy",\n    "component": "Orion-Tr 48/12-30 360W auxiliary converter, research candidate only",\n    "source_url": "https://www.victronenergy.com/media/pricelist/Pricelist_Victron_EUR_C_2026-Q2_web.pdf",\n    "accessed_utc": "2026-10-09",\n    "primary_source": true,\n    "dimensions_mm_HWD": [\n      130,\n      186,\n      80\n    ],\n    "mass_kg": 1.8,\n    "source_CAD_acquired": false,\n    "model_class": "not modeled, unselected candidate",\n    "redistribution": "Source link only, no supplier geometry",\n    "limits": [\n      "No electrical selection or purchase.",\n      "Exact converter supply range, cooling, isolation and transient compatibility must be verified before modeling integration."\n    ]\n  },\n  {\n    "id": "WD220-CANDIDATE",\n    "manufacturer": "e-comer / Benevelli",\n    "component": "SMAC132-050,48V,EM-brake,5-studs",\n    "source_url": "https://www.e-comer.com/en/?Itemid=461",\n    "drawing_part": "99.220.041.A.3.39",\n    "source_CAD_acquired_by_other_worker": true,\n    "source_CAD_included_in_this_package": false,\n    "model_class": "unmodeled traction candidate; factual rating summary only",\n    "redistribution": "RESTRICTED supplier geometry; no STEP/PDF/derived mesh in this body/power package",\n    "facts": {\n      "candidate_quantity": 6,\n      "voltage_V": 48,\n      "rated_power_W_each": 2200,\n      "duty": "S2,60 minutes; NOT S1 continuous",\n      "rated_current_A_each": 47,\n      "six_drive_arithmetic_current_A": 282,\n      "drive_unit_price": null,\n      "currency": null,\n      "mass_kg_each": null,\n      "rated_current_A_each_basis": "Motor-block rating in supplier drawing; not explicitly DC-bus input current",\n      "six_drive_arithmetic_current_basis": "Motor-rating sum only; do not use as battery demand",\n      "six_drive_mechanical_rating_sum_W": 13200,\n      "ideal_DC_current_at_51_2V_for_simultaneous_rated_output_A": 257.8125\n    },\n    "limits": [\n      "47A each is not explicitly DC-bus current; inverter input/efficiency data are required.",\n      "Six motor current ratings sum282A arithmetically but cannot be directly compared as pack demand.",\n      "If all six deliver2.2kW mechanical output simultaneously,13.2kW/51.2V gives257.8A even at ideal100% efficiency, before arms/auxiliaries; actual required input is higher.",\n      "That simultaneous-rated-output scenario exceeds four batteries\' summed individually recommended180A, but does not by itself exceed summed400A individual maxima. Neither sum is a qualified pack rating.",\n      "S2 60min duty is not continuous.",\n      "BMS current sharing, inverter selection, starting inrush, regeneration, fuse/contactors, wiring, pack thermal design and mission current limits remain unqualified.",\n      "Motor/traction controller/tire/rim/harness/power protection/cooling masses and drive costs are excluded/unknown in this package; not zero."\n    ]\n  }\n]')


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, str(P/'build_body_power.py')], check=True)


In [ ]:
import json
r=json.loads((P/'verification.json').read_text())
assert r['individual_valid']
assert not r['positive_volume_body_intersections']
assert not r['frame_intersections']
assert not r['dimensional_reservation_intersections']
print({k:r[k] for k in ['original_part_count','bounds_m','body_hardware_mass_kg','body_hardware_com_m']})
print('Geometric checks passed; no fabrication, electrical, motion, IP or safety qualification implied.')


In [ ]:
import shutil
archive=shutil.make_archive('/content/body_power_R01_outputs','zip',P)
from google.colab import files
files.download(archive)
